# Building Silver Tables

### Reading Bronze Tables

In [0]:
%sql
USE CATALOG merchai_data;
USE SCHEMA bronze;

In [0]:
orders_df = spark.table("orders")
products_df = spark.table("products")
departments_df = spark.table("departments")
aisles_df = spark.table("aisles")
prior_df = spark.table("order_products_prior")

In [0]:
display(orders_df.limit(10))
display(products_df.limit(10))
display(departments_df.limit(10))
display(aisles_df.limit(10))
display(prior_df.limit(10))

### Drop Duplicates

In [0]:
orders_df = orders_df.dropDuplicates(["order_id"])

products_df = products_df.dropDuplicates(["product_id"])

departments_df = departments_df.dropDuplicates(["department_id"])

aisles_df = aisles_df.dropDuplicates(["aisle_id"])

prior_df = prior_df.dropDuplicates(["order_id", "product_id"])

### Check Null values

In [0]:
from pyspark.sql.functions import col, sum, when

for name, df in [
    ("orders", orders_df),
    ("products", products_df),
    ("departments", departments_df),
    ("aisles", aisles_df),
    ("prior", prior_df)
]:
    print(f"\n{name.upper()}")

    display(
        df.select([
            sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
            for c in df.columns
        ])
    )

### Handling null values

In [0]:
orders_df = orders_df.fillna({
    "days_since_prior_order":0
})

In [0]:
products_df = products_df.dropna()

## Join Tables

### Orders + Prior + Products + Departments + Aisles + 

In [0]:
silver_df = prior_df.join(
    orders_df,
    on="order_id",
    how="inner"
)

silver_df = silver_df.join(
    products_df,
    on="product_id",
    how="left"
)

silver_df = silver_df.join(
    departments_df,
    on="department_id",
    how="left"
)

silver_df = silver_df.join(
    aisles_df,
    on="aisle_id",
    how="left"
)

### Test Joins

In [0]:
display(silver_df.limit(10))

### checking schema

In [0]:
silver_df.printSchema()

## Writing Silver Table

In [0]:
%sql
USE CATALOG merchai_data;
USE SCHEMA silver;

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_customer_orders")